# Resolution transitions in `nordic_baltic_raw.parquet`

Two questions:

1. **When does each series switch from 60-minute to 15-minute resolution?**
2. **When does the 15-minute data stop being the hourly value repeated four times, and start carrying
   genuine intra-hour variation?**

Plus a cleaning cell (section 6) that writes a model-ready hourly panel.

Everything is computed from the raw file — no hard-coded dates.

## 0. Setup

In [18]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

# Resolve the raw file relative to the notebook, wherever it is opened from.
CANDIDATES = [
    Path("datasets/nordic_baltic_raw.parquet"),
    Path("../datasets/nordic_baltic_raw.parquet"),
    Path("../../datasets/nordic_baltic_raw.parquet"),
]
RAW = next((p for p in CANDIDATES if p.exists()), None)
if RAW is None:
    raise FileNotFoundError(
        "nordic_baltic_raw.parquet not found. Set RAW manually to its path."
    )

OUT = RAW.parent / "nordic_baltic_clean_hourly.parquet"
print("raw   :", RAW.resolve())
print("output:", OUT.resolve())

raw   : C:\Users\Frede\OneDrive\Skrivebord\Masters Thesis v2\Code\EPF_Masters\datasets\nordic_baltic_raw.parquet
output: C:\Users\Frede\OneDrive\Skrivebord\Masters Thesis v2\Code\EPF_Masters\datasets\nordic_baltic_clean_hourly.parquet


In [19]:
KEY = ["zone", "variable", "psr_type", "resolution", "timestamp_utc"]
CAT = ["zone", "eic", "variable", "document_type", "process_type", "resolution",
       "psr_type", "business_type", "curve_type", "contract_type", "unit", "currency"]

# psrType codes carried by the wind & solar forecast document (A69). Spelled out at load time so
# every table below - and the parquet written in section 6 - reads in words rather than codes.
# Unlisted codes pass through as themselves, so a new production type is visible, not silently lost.
PSR_NAMES = {"B16": "solar", "B18": "wind_offshore", "B19": "wind_onshore"}


def load_raw(path=RAW) -> pd.DataFrame:
    """Read the parquet and normalise the dictionary-encoded columns to plain strings.

    Nulls become empty strings on purpose: pandas treats NaN keys as *equal* in
    duplicated()/drop_duplicates(), which silently collapses every price row (psr_type is
    null for prices) into one group. Empty strings behave correctly.
    """
    df = pq.read_table(path).to_pandas()
    for c in CAT:
        if c in df.columns:
            df[c] = df[c].astype("object").fillna("").astype(str)
    df["psr_type"] = df["psr_type"].replace(PSR_NAMES)
    df["series"] = (
        df["zone"] + " | " + df["variable"]
        + np.where(df["psr_type"] != "", " [" + df["psr_type"] + "]", "")
    )
    return df


raw = load_raw()
print(f"{len(raw):,} rows x {raw.shape[1]} columns, {raw['series'].nunique()} series")
print(f"span: {raw.timestamp_utc.min()} -> {raw.timestamp_utc.max()}")
raw.head(3)

5,883,021 rows x 15 columns, 67 series
span: 2015-12-31 23:00:00+00:00 -> 2025-10-02 21:45:00+00:00


,zone,eic,variable,document_type,process_type,timestamp_utc,value,resolution,psr_type,business_type,curve_type,contract_type,unit,currency,series
0,DK1,10YDK-1--------W,price,A44,,2015-12-31 23:00:00+00:00,16.39,PT60M,,A62,A03,A01,MWH,EUR,DK1 | price
1,DK1,10YDK-1--------W,price,A44,,2016-01-01 00:00:00+00:00,16.04,PT60M,,A62,A03,A01,MWH,EUR,DK1 | price
2,DK1,10YDK-1--------W,price,A44,,2016-01-01 01:00:00+00:00,15.74,PT60M,,A62,A03,A01,MWH,EUR,DK1 | price


## 1. Duplicate rows

Check before anything else: a duplicated key breaks `pivot`/`unstack`, doubles rows on a merge, and
silently double-weights part of the sample in a loss function.

In [20]:
counts = raw.groupby(KEY, observed=True).size()
print("multiplicity of each (zone, variable, psr_type, resolution, timestamp) key:")
print(counts.value_counts().sort_index().to_string())

n_dupe = len(raw) - len(raw.drop_duplicates(KEY))
print(f"\nduplicate rows: {n_dupe:,} of {len(raw):,} ({100 * n_dupe / len(raw):.2f}%)")

# Are the duplicates identical, or do they disagree? Only the second case is lossy to drop.
conflicting = raw.groupby(KEY, observed=True)["value"].nunique()
print(f"keys whose duplicates carry CONFLICTING values: {(conflicting > 1).sum():,}")
print("-> drop_duplicates() is lossless" if (conflicting > 1).sum() == 0
      else "-> WARNING: investigate before dropping")

multiplicity of each (zone, variable, psr_type, resolution, timestamp) key:
1    5791581
2      45720

duplicate rows: 45,720 of 5,883,021 (0.78%)
keys whose duplicates carry CONFLICTING values: 0
-> drop_duplicates() is lossless


In [21]:
# Which variables are affected, and where in the calendar do the duplicates sit?
SUBKEY = [c for c in KEY if c != "variable"]
by_var = raw.groupby("variable").apply(
    lambda g: pd.Series({"rows": len(g), "dupes": len(g) - len(g.drop_duplicates(SUBKEY))}),
    include_groups=False,
)
by_var["pct"] = (100 * by_var.dupes / by_var.rows).round(2)
print(by_var.to_string(), "\n")

dup_mask = raw.duplicated(KEY, keep=False)
if dup_mask.any():
    d = raw.loc[dup_mask].copy()
    d["local_day"] = d.timestamp_utc.dt.tz_convert("Europe/Copenhagen").dt.day
    print("day-of-month of duplicated stamps:")
    print(d.local_day.value_counts().head(10).to_string())
    print("\nexample duplicate pair:")
    ts, z = d.iloc[0].timestamp_utc, d.iloc[0].zone
    display(raw[(raw.timestamp_utc == ts) & (raw.zone == z) & (raw.variable == "price")])

                        rows  dupes   pct
variable                                 
generation_forecast  3035827      0  0.00
load_forecast        1437464      0  0.00
price                1409730  45720  3.24 

day-of-month of duplicated stamps:
local_day
1    91440

example duplicate pair:


,zone,eic,variable,document_type,process_type,timestamp_utc,value,resolution,psr_type,business_type,curve_type,contract_type,unit,currency,series
744,DK1,10YDK-1--------W,price,A44,,2016-01-31 23:00:00+00:00,16.31,PT60M,,A62,A03,A01,MWH,EUR,DK1 | price
768,DK1,10YDK-1--------W,price,A44,,2016-01-31 23:00:00+00:00,16.31,PT60M,,A62,A03,A01,MWH,EUR,DK1 | price


**Reading it:** the duplicates are confined to `price`, every affected key appears exactly twice, the
two rows are byte-identical, and they land on the 1st of each local month. That is a month-chunked download
with inclusive endpoints on both ends

In [22]:
df = raw.drop_duplicates(KEY, keep="first").copy()
print(f"deduped: {len(df):,} rows")

deduped: 5,837,301 rows


## 2. Zero-variance and near-zero variance series

A column with `nunique() == 1` has zero variance. Standardising it yields `NaN` for every entry (or a
column of zeros under `StandardScaler`), and it is perfectly collinear
with the intercept in any LEAR/LASSO benchmark.

In [23]:
def series_health(d: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for s, g in d.groupby("series", sort=True):
        v = g["value"]
        u = np.sort(v.dropna().unique())
        steps = np.diff(u)
        rows.append({
            "series": s,
            "n": len(v),
            "n_distinct": v.nunique(),
            "pct_zero": round(100 * (v.abs() < 1e-9).mean(), 1),
            "pct_nan": round(100 * v.isna().mean(), 1),
            "mean": round(v.mean(), 2),
            "max": round(v.max(), 1),
            "min_step": round(steps[steps > 0].min(), 3) if len(u) > 1 else np.nan,
            "first": g.timestamp_utc.min().date(),
            "last": g.timestamp_utc.max().date(),
        })
    return pd.DataFrame(rows)


health = series_health(df)
dead = health[health.n_distinct == 1]
print(f"CONSTANT series (nunique == 1): {len(dead)}, "
      f"{dead.n.sum():,} rows = {100 * dead.n.sum() / len(df):.1f}% of the deduped file")
display(dead)

CONSTANT series (nunique == 1): 9, 675,239 rows = 11.6% of the deduped file


,series,n,n_distinct,pct_zero,pct_nan,mean,max,min_step,first,last
26,NO1 | generation_forecast [solar],99763,1,100.0,0.0,0.0,0.0,NaN,2016-01-01,2025-10-01
27,NO1 | generation_forecast [wind_offshore],44106,1,100.0,0.0,0.0,0.0,NaN,2016-01-01,2024-01-19
31,NO2 | generation_forecast [solar],99763,1,100.0,0.0,0.0,0.0,NaN,2016-01-01,2025-10-01
36,NO3 | generation_forecast [solar],99763,1,100.0,0.0,0.0,0.0,NaN,2016-01-01,2025-10-01
37,NO3 | generation_forecast [wind_offshore],44106,1,100.0,0.0,0.0,0.0,NaN,2016-01-01,2024-01-19
41,NO4 | generation_forecast [solar],99763,1,100.0,0.0,0.0,0.0,NaN,2016-01-01,2025-10-01
42,NO4 | generation_forecast [wind_offshore],44106,1,100.0,0.0,0.0,0.0,NaN,2016-01-01,2024-01-19
46,NO5 | generation_forecast [solar],99763,1,100.0,0.0,0.0,0.0,NaN,2016-01-01,2025-10-01
47,NO5 | generation_forecast [wind_offshore],44106,1,100.0,0.0,0.0,0.0,NaN,2016-01-01,2024-01-19


In [24]:
near = health[(health.n_distinct > 1) & (health.pct_zero > 90)]
print("NEAR-constant series (>90% zero but not exactly constant) — decide case by case:")
display(near)

print("For contrast, legitimately zero-heavy solar series (night hours):")
display(health[health.series.str.contains(r"\[solar\]") & (health.n_distinct > 1000)]
        [["series", "n_distinct", "pct_zero", "mean", "max"]])

NEAR-constant series (>90% zero but not exactly constant) — decide case by case:


,series,n,n_distinct,pct_zero,pct_nan,mean,max,min_step,first,last
32,NO2 | generation_forecast [wind_offshore],46226,47,95.8,0.0,0.12,4.6,0.10,2016-01-01,2025-10-01
48,NO5 | generation_forecast [wind_onshore],99763,915,98.4,0.0,0.11,19.1,0.01,2016-01-01,2025-10-01


For contrast, legitimately zero-heavy solar series (night hours):


,series,n_distinct,pct_zero,mean,max
0,DK1 | generation_forecast [solar],28536,35.3,198.44,2368.3
5,DK2 | generation_forecast [solar],23852,38.3,88.36,876.7
10,EE | generation_forecast [solar],4134,22.2,52.40,605.9
14,FI | generation_forecast [solar],14492,44.8,107.78,998.9
18,LT | generation_forecast [solar],14712,49.1,84.31,1517.5
22,LV | generation_forecast [solar],1938,37.0,30.06,266.6
55,SE2 | generation_forecast [solar],4354,41.5,6.37,72.6
59,SE3 | generation_forecast [solar],12867,17.4,106.32,1180.4
63,SE4 | generation_forecast [solar],11899,22.2,70.57,848.6


In [25]:
# NO2 [wind_offshore] is 95.8% zero with only 47 distinct values, NO5 [wind_onshore] is 98.4%. Neither is exactly
# constant, so an nunique() filter keeps them - but both are constant for every practical purpose:
# a near-zero MAD makes the scaler blow the column up, and there is no signal for a LEAR/LASSO
# benchmark to fit. The same threshold also removes the 9 exactly-constant series, which are 100% zero.
#
# 90% is safe for solar: the zero-heaviest legitimate solar series peaks at 49% zero (contrast table
# above), so nothing genuine sits anywhere near the cut.
MAX_PCT_ZERO = 90.0


def drop_near_constant(d: pd.DataFrame, max_pct_zero=MAX_PCT_ZERO, verbose=True):
    """Drop every series that is more than `max_pct_zero` percent exactly zero."""
    log = lambda *a: print(*a) if verbose else None
    zero_share = 100 * (d["value"].abs() < 1e-9).groupby(d["series"]).transform("mean")
    kill = zero_share > max_pct_zero
    dropped = sorted(d.loc[kill, "series"].unique())

    n = len(d)
    log(f"dropped {len(dropped)} series >{max_pct_zero:g}% zero: "
        f"{n:,} -> {n - int(kill.sum()):,} rows  (-{int(kill.sum()):,})")
    for name in dropped:
        v = d.loc[d.series == name, "value"]
        log(f"     {name:<32} {100 * (v.abs() < 1e-9).mean():5.1f}% zero, {v.nunique():>5} distinct, "
            f"max {v.max():.1f}")
    return d[~kill].copy(), dropped


df, dropped_series = drop_near_constant(df)
print(f"{df.series.nunique()} series left for the rest of the notebook")


dropped 11 series >90% zero: 5,837,301 -> 5,016,073 rows  (-821,228)
     NO1 | generation_forecast [solar] 100.0% zero,     1 distinct, max 0.0
     NO1 | generation_forecast [wind_offshore] 100.0% zero,     1 distinct, max 0.0
     NO2 | generation_forecast [solar] 100.0% zero,     1 distinct, max 0.0
     NO2 | generation_forecast [wind_offshore]  95.8% zero,    47 distinct, max 4.6
     NO3 | generation_forecast [solar] 100.0% zero,     1 distinct, max 0.0
     NO3 | generation_forecast [wind_offshore] 100.0% zero,     1 distinct, max 0.0
     NO4 | generation_forecast [solar] 100.0% zero,     1 distinct, max 0.0
     NO4 | generation_forecast [wind_offshore] 100.0% zero,     1 distinct, max 0.0
     NO5 | generation_forecast [solar] 100.0% zero,     1 distinct, max 0.0
     NO5 | generation_forecast [wind_offshore] 100.0% zero,     1 distinct, max 0.0
     NO5 | generation_forecast [wind_onshore]  98.4% zero,   915 distinct, max 19.1
56 series left for the rest of the notebook


## 3. When does each series switch from PT60M to PT15M?

All timestamps are UTC. Local time is UTC+1/+2 (CET/CEST) for DK/NO/SE and UTC+2/+3 (EET/EEST) for
FI/EE/LV/LT, so a 23:00 or 22:00 UTC boundary is midnight local.

In [26]:
def transitions(d: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for s, g in d.groupby("series", sort=True):
        g60, g15 = g[g.resolution == "PT60M"], g[g.resolution == "PT15M"]
        overlap = (
            pd.Index(g60.timestamp_utc).intersection(pd.Index(g15.timestamp_utc))
            if len(g60) and len(g15) else pd.Index([])
        )
        rows.append({
            "series": s,
            "n_60min": len(g60),
            "n_15min": len(g15),
            "first_60min": g60.timestamp_utc.min() if len(g60) else pd.NaT,
            "last_60min": g60.timestamp_utc.max() if len(g60) else pd.NaT,
            "first_15min": g15.timestamp_utc.min() if len(g15) else pd.NaT,
            "last_15min": g15.timestamp_utc.max() if len(g15) else pd.NaT,
            "overlap_stamps": len(overlap),
        })
    return pd.DataFrame(rows)


trans = transitions(df)
print("timestamps carrying BOTH resolutions:", int(trans.overlap_stamps.sum()),
      "-> every switch is a clean cutover\n")
display(trans[trans.series.str.contains("price")])

timestamps carrying BOTH resolutions: 0 -> every switch is a clean cutover



,series,n_60min,n_15min,first_60min,last_60min,first_15min,last_15min,overlap_stamps
4,DK1 | price,85463,192,2015-12-31 23:00:00+00:00,2025-09-30 21:00:00+00:00,2025-09-30 22:00:00+00:00,2025-10-02 21:45:00+00:00,0
9,DK2 | price,85463,192,2015-12-31 23:00:00+00:00,2025-09-30 21:00:00+00:00,2025-09-30 22:00:00+00:00,2025-10-02 21:45:00+00:00,0
13,EE | price,85463,192,2015-12-31 23:00:00+00:00,2025-09-30 21:00:00+00:00,2025-09-30 22:00:00+00:00,2025-10-02 21:45:00+00:00,0
17,FI | price,85463,192,2015-12-31 23:00:00+00:00,2025-09-30 21:00:00+00:00,2025-09-30 22:00:00+00:00,2025-10-02 21:45:00+00:00,0
21,LT | price,85463,192,2015-12-31 23:00:00+00:00,2025-09-30 21:00:00+00:00,2025-09-30 22:00:00+00:00,2025-10-02 21:45:00+00:00,0
25,LV | price,85463,192,2015-12-31 23:00:00+00:00,2025-09-30 21:00:00+00:00,2025-09-30 22:00:00+00:00,2025-10-02 21:45:00+00:00,0
28,NO1 | price,80184,21308,2015-12-31 23:00:00+00:00,2025-05-16 21:00:00+00:00,2025-02-20 23:00:00+00:00,2025-10-02 21:45:00+00:00,0
31,NO2 | price,80184,21308,2015-12-31 23:00:00+00:00,2025-05-16 21:00:00+00:00,2025-02-20 23:00:00+00:00,2025-10-02 21:45:00+00:00,0
34,NO3 | price,80184,21308,2015-12-31 23:00:00+00:00,2025-05-16 21:00:00+00:00,2025-02-20 23:00:00+00:00,2025-10-02 21:45:00+00:00,0
37,NO4 | price,80184,21308,2015-12-31 23:00:00+00:00,2025-05-16 21:00:00+00:00,2025-02-20 23:00:00+00:00,2025-10-02 21:45:00+00:00,0


In [27]:
display(trans[trans.series.str.contains("load_forecast")])
display(trans[trans.series.str.contains("generation_forecast")])

never = trans[trans.n_15min == 0]
print(f"\nseries that never reach 15-min inside this file: {len(never)}")
print(sorted(never.series.tolist()))

,series,n_60min,n_15min,first_60min,last_60min,first_15min,last_15min,overlap_stamps
3,DK1 | load_forecast,85488,0,2016-01-01 00:00:00+00:00,2025-10-01 23:00:00+00:00,NaT,NaT,0
8,DK2 | load_forecast,85488,0,2016-01-01 00:00:00+00:00,2025-10-01 23:00:00+00:00,NaT,NaT,0
12,EE | load_forecast,85056,0,2016-01-01 00:00:00+00:00,2025-10-01 23:00:00+00:00,NaT,NaT,0
16,FI | load_forecast,64656,83040,2016-01-01 00:00:00+00:00,2023-05-20 23:00:00+00:00,2023-05-21 00:00:00+00:00,2025-10-01 23:45:00+00:00,0
20,LT | load_forecast,77269,30620,2016-01-01 00:00:00+00:00,2024-11-16 23:00:00+00:00,2024-11-17 00:00:00+00:00,2025-10-01 23:45:00+00:00,0
24,LV | load_forecast,85271,0,2016-01-01 00:00:00+00:00,2025-10-01 23:00:00+00:00,NaT,NaT,0
27,NO1 | load_forecast,80736,19008,2016-01-01 00:00:00+00:00,2025-03-17 23:00:00+00:00,2025-03-18 00:00:00+00:00,2025-10-01 23:45:00+00:00,0
30,NO2 | load_forecast,80736,19008,2016-01-01 00:00:00+00:00,2025-03-17 23:00:00+00:00,2025-03-18 00:00:00+00:00,2025-10-01 23:45:00+00:00,0
33,NO3 | load_forecast,80736,19008,2016-01-01 00:00:00+00:00,2025-03-17 23:00:00+00:00,2025-03-18 00:00:00+00:00,2025-10-01 23:45:00+00:00,0
36,NO4 | load_forecast,80736,19008,2016-01-01 00:00:00+00:00,2025-03-17 23:00:00+00:00,2025-03-18 00:00:00+00:00,2025-10-01 23:45:00+00:00,0


,series,n_60min,n_15min,first_60min,last_60min,first_15min,last_15min,overlap_stamps
0,DK1 | generation_forecast [solar],80904,16800,2016-01-01 00:00:00+00:00,2025-04-07 23:00:00+00:00,2025-04-08 00:00:00+00:00,2025-10-01 23:45:00+00:00,0
1,DK1 | generation_forecast [wind_offshore],81096,16608,2016-01-01 00:00:00+00:00,2025-04-07 23:00:00+00:00,2025-04-08 00:00:00+00:00,2025-10-01 23:45:00+00:00,0
2,DK1 | generation_forecast [wind_onshore],80976,16629,2016-01-01 00:00:00+00:00,2025-04-07 23:00:00+00:00,2025-04-08 00:00:00+00:00,2025-10-01 23:45:00+00:00,0
5,DK2 | generation_forecast [solar],80952,16792,2016-01-01 00:00:00+00:00,2025-04-07 23:00:00+00:00,2025-04-08 00:00:00+00:00,2025-10-01 23:45:00+00:00,0
6,DK2 | generation_forecast [wind_offshore],81120,16608,2016-01-01 00:00:00+00:00,2025-04-07 23:00:00+00:00,2025-04-08 00:00:00+00:00,2025-10-01 23:45:00+00:00,0
7,DK2 | generation_forecast [wind_onshore],81000,16608,2016-01-01 00:00:00+00:00,2025-04-07 23:00:00+00:00,2025-04-08 00:00:00+00:00,2025-10-01 23:45:00+00:00,0
10,EE | generation_forecast [solar],44259,0,2020-09-01 21:00:00+00:00,2025-10-01 23:00:00+00:00,NaT,NaT,0
11,EE | generation_forecast [wind_onshore],85175,0,2016-01-01 00:00:00+00:00,2025-10-01 23:00:00+00:00,NaT,NaT,0
14,FI | generation_forecast [solar],33051,82944,2019-07-05 21:00:00+00:00,2023-05-20 23:00:00+00:00,2023-05-21 00:00:00+00:00,2025-10-01 23:45:00+00:00,0
15,FI | generation_forecast [wind_onshore],64542,83040,2016-01-01 00:00:00+00:00,2023-05-20 23:00:00+00:00,2023-05-21 00:00:00+00:00,2025-10-01 23:45:00+00:00,0



series that never reach 15-min inside this file: 18
['DK1 | load_forecast', 'DK2 | load_forecast', 'EE | generation_forecast [solar]', 'EE | generation_forecast [wind_onshore]', 'EE | load_forecast', 'LV | load_forecast', 'SE1 | generation_forecast [solar]', 'SE1 | generation_forecast [wind_onshore]', 'SE1 | load_forecast', 'SE2 | generation_forecast [solar]', 'SE2 | generation_forecast [wind_onshore]', 'SE2 | load_forecast', 'SE3 | generation_forecast [solar]', 'SE3 | generation_forecast [wind_onshore]', 'SE3 | load_forecast', 'SE4 | generation_forecast [solar]', 'SE4 | generation_forecast [wind_onshore]', 'SE4 | load_forecast']


## 4. When does 15-min data stop being the hourly value repeated 4×?

**Method.** For every clock hour that has exactly four quarter-hour observations, flag it as *varying* if the
four values are not all identical. Hours in which the series is identically zero (night-time solar) are
excluded from the `pct_active` statistic, since they would otherwise count as spurious "flat" hours.

*First sustained day* = the first day from which at least 50% of active hours vary, holding for at least
three consecutive days. This distinguishes a real switch from a single stray hour.

In [28]:
def intra_hour_variation(d: pd.DataFrame, min_frac=0.5, window=7, min_days=3) -> pd.DataFrame:
    q = d[d.resolution == "PT15M"].copy()
    q["hour"] = q.timestamp_utc.dt.floor("h")
    rows = []
    for s, g in q.groupby("series", sort=True):
        a = g.groupby("hour")["value"].agg(nun="nunique", cnt="count", mx="max")
        a = a[a.cnt == 4]                       # ignore DST-short and partial hours
        if a.empty:
            continue
        a["varies"] = a.nun > 1
        active = a[a.mx.abs() > 1e-9]           # drop all-zero (night) hours
        daily = (active.groupby(active.index.date)["varies"].mean()
                 if len(active) else pd.Series(dtype=float))
        daily.index = pd.to_datetime(daily.index)

        sustained = pd.NaT
        for i, day in enumerate(daily.index):
            w = daily.iloc[i:i + window]
            if len(w) >= min_days and (w >= min_frac).all():
                sustained = day
                break

        first15 = g.timestamp_utc.min()
        rows.append({
            "series": s,
            "first_15min": first15,
            "pct_hours_varying": round(100 * a.varies.mean(), 1),
            "pct_active_varying": round(100 * active.varies.mean(), 1) if len(active) else np.nan,
            "pct_hours_all_zero": round(100 * (1 - len(active) / len(a)), 1),
            "first_varying_hour": a.index[a.varies].min() if a.varies.any() else pd.NaT,
            "first_sustained_day": sustained,
            "lag_days": (sustained - first15.tz_localize(None).normalize()).days
                        if pd.notna(sustained) else np.nan,
        })
    return pd.DataFrame(rows)


var = intra_hour_variation(df)
display(var[var.series.str.contains("price")])

,series,first_15min,pct_hours_varying,pct_active_varying,pct_hours_all_zero,first_varying_hour,first_sustained_day,lag_days
3,DK1 | price,2025-09-30 22:00:00+00:00,100.0,100.0,0.0,2025-09-30 22:00:00+00:00,2025-09-30,0.0
7,DK2 | price,2025-09-30 22:00:00+00:00,100.0,100.0,0.0,2025-09-30 22:00:00+00:00,2025-09-30,0.0
8,EE | price,2025-09-30 22:00:00+00:00,97.9,97.9,0.0,2025-09-30 22:00:00+00:00,2025-09-30,0.0
12,FI | price,2025-09-30 22:00:00+00:00,100.0,100.0,0.0,2025-09-30 22:00:00+00:00,2025-09-30,0.0
16,LT | price,2025-09-30 22:00:00+00:00,100.0,100.0,0.0,2025-09-30 22:00:00+00:00,2025-09-30,0.0
19,LV | price,2025-09-30 22:00:00+00:00,100.0,100.0,0.0,2025-09-30 22:00:00+00:00,2025-09-30,0.0
22,NO1 | price,2025-02-20 23:00:00+00:00,0.9,0.9,0.0,2025-09-30 22:00:00+00:00,NaT,NaN
25,NO2 | price,2025-02-20 23:00:00+00:00,0.9,0.9,0.0,2025-09-30 22:00:00+00:00,NaT,NaN
28,NO3 | price,2025-02-20 23:00:00+00:00,0.9,0.9,0.2,2025-09-30 22:00:00+00:00,NaT,NaN
31,NO4 | price,2025-02-20 23:00:00+00:00,0.9,0.9,0.4,2025-09-30 22:00:00+00:00,NaT,NaN


**The headline.** Every zone gets genuinely varying 15-minute prices at the same instant —
`2025-09-30 22:00 UTC`, i.e. `2025-10-01 00:00` local, the EU-wide 15-minute MTU go-live.

Norway is the trap: it started publishing 15-minute *stamps* on 2025-02-21, seven months early, but those
are the hourly price copied four times. Under 1% of Norwegian 15-minute hours vary, and all of them are on
1–2 October.

In [29]:
# Proof: the same Norwegian hour before and after the go-live.
no1 = df[(df.series == "NO1 | price") & (df.resolution == "PT15M")]


def window(start, hours=2):
    t0 = pd.Timestamp(start, tz="UTC")
    w = no1[(no1.timestamp_utc >= t0) & (no1.timestamp_utc < t0 + pd.Timedelta(hours=hours))]
    return w.set_index("timestamp_utc")["value"]


print("NO1, 2025-06-10 (15-min stamps, hourly content):")
print(window("2025-06-10 08:00").to_string())
print("\nNO1, 2025-10-01 (genuine 15-min):")
print(window("2025-10-01 08:00").to_string())

NO1, 2025-06-10 (15-min stamps, hourly content):
timestamp_utc
2025-06-10 08:00:00+00:00    54.21
2025-06-10 08:15:00+00:00    54.21
2025-06-10 08:30:00+00:00    54.21
2025-06-10 08:45:00+00:00    54.21
2025-06-10 09:00:00+00:00    49.20
2025-06-10 09:15:00+00:00    49.20
2025-06-10 09:30:00+00:00    49.20
2025-06-10 09:45:00+00:00    49.20

NO1, 2025-10-01 (genuine 15-min):
timestamp_utc
2025-10-01 08:00:00+00:00    60.57
2025-10-01 08:15:00+00:00    55.68
2025-10-01 08:30:00+00:00    55.55
2025-10-01 08:45:00+00:00    58.70
2025-10-01 09:00:00+00:00    51.96
2025-10-01 09:15:00+00:00    55.89
2025-10-01 09:30:00+00:00    58.49
2025-10-01 09:45:00+00:00    57.07


In [30]:
display(var[~var.series.str.contains("price")])

print("Monthly share of varying active hours — the Baltic ramp-ups are slow and incomplete:")
q = df[df.resolution == "PT15M"].copy()
q["hour"] = q.timestamp_utc.dt.floor("h")
for s in ["LT | generation_forecast [wind_onshore]", "LV | generation_forecast [wind_onshore]",
          "FI | load_forecast", "DK1 | generation_forecast [wind_onshore]"]:
    g = q[q.series == s]
    if g.empty:
        continue
    a = g.groupby("hour")["value"].agg(nun="nunique", cnt="count", mx="max")
    a = a[(a.cnt == 4) & (a.mx.abs() > 1e-9)]
    m = a.assign(v=a.nun > 1).groupby(a.index.tz_localize(None).to_period("M"))["v"].mean()
    print(f"\n{s}")
    print(m.round(3).to_string())

,series,first_15min,pct_hours_varying,pct_active_varying,pct_hours_all_zero,first_varying_hour,first_sustained_day,lag_days
0,DK1 | generation_forecast [solar],2025-04-08 00:00:00+00:00,85.0,92.4,8.0,2025-04-10 01:00:00+00:00,2025-04-10,2.0
1,DK1 | generation_forecast [wind_offshore],2025-04-08 00:00:00+00:00,99.3,99.4,0.0,2025-04-09 22:00:00+00:00,2025-04-09,1.0
2,DK1 | generation_forecast [wind_onshore],2025-04-08 00:00:00+00:00,99.4,99.4,0.0,2025-04-09 00:00:00+00:00,2025-04-09,1.0
4,DK2 | generation_forecast [solar],2025-04-08 00:00:00+00:00,80.5,92.2,12.7,2025-04-10 03:00:00+00:00,2025-04-10,2.0
5,DK2 | generation_forecast [wind_offshore],2025-04-08 00:00:00+00:00,99.1,99.1,0.1,2025-04-09 22:00:00+00:00,2025-04-09,1.0
6,DK2 | generation_forecast [wind_onshore],2025-04-08 00:00:00+00:00,98.0,98.0,0.0,2025-04-09 22:00:00+00:00,2025-04-09,1.0
9,FI | generation_forecast [solar],2023-05-21 00:00:00+00:00,61.8,99.5,37.9,2023-05-22 00:00:00+00:00,2023-05-22,1.0
10,FI | generation_forecast [wind_onshore],2023-05-21 00:00:00+00:00,99.1,99.1,0.0,2023-05-21 22:00:00+00:00,2023-05-22,1.0
11,FI | load_forecast,2023-05-21 00:00:00+00:00,99.1,99.1,0.0,2023-05-21 22:00:00+00:00,2023-05-22,1.0
13,LT | generation_forecast [solar],2024-10-08 00:00:00+00:00,46.7,82.2,43.2,2024-11-22 06:00:00+00:00,2024-11-25,48.0


Monthly share of varying active hours — the Baltic ramp-ups are slow and incomplete:

LT | generation_forecast [wind_onshore]
hour
2024-10    0.000
2024-11    0.104
2024-12    0.349
2025-01    0.324
2025-02    1.000
2025-03    0.999
2025-04    1.000
2025-05    1.000
2025-06    1.000
2025-07    1.000
2025-08    1.000
2025-09    1.000
2025-10    1.000
Freq: M

LV | generation_forecast [wind_onshore]
hour
2024-12    0.682
2025-01    0.723
2025-02    0.574
2025-03    0.120
2025-04    0.717
2025-05    0.883
2025-06    0.883
2025-07    0.000
2025-08    0.293
2025-09    0.590
2025-10    0.708
Freq: M

FI | load_forecast
hour
2023-05    0.917
2023-06    1.000
2023-07    1.000
2023-08    1.000
2023-09    1.000
2023-10    1.000
2023-11    1.000
2023-12    1.000
2024-01    1.000
2024-02    1.000
2024-03    1.000
2024-04    1.000
2024-05    1.000
2024-06    1.000
2024-07    1.000
2024-08    1.000
2024-09    1.000
2024-10    1.000
2024-11    1.000
2024-12    0.976
2025-01    0.972
2025-02    0.984


**Reading it.** For FI, NO and DK the resolution change and the arrival of real sub-hourly content are
effectively the same event (1–2 day lag). Lithuania published flat quarter-hours for 1–3.5 months first, and
Latvia never stabilises — LV generation drops back to 0% varying for whole months in 2025. Treat Latvian
generation forecasts as unreliable at 15-minute granularity.

## 5. Summary table

One row per series, joining everything above. Saved next to the raw file.

In [31]:
summary = (trans
           .merge(var.drop(columns=["first_15min"]), on="series", how="left")
           .merge(health[["series", "n_distinct", "pct_zero", "mean", "max"]], on="series", how="left"))
summary.insert(1, "zone", summary.series.str.split(" | ", regex=False).str[0])

summary_path = RAW.parent / "resolution_summary.csv"
summary.to_csv(summary_path, index=False)
print("written:", summary_path.resolve())
display(summary)

written: C:\Users\Frede\OneDrive\Skrivebord\Masters Thesis v2\Code\EPF_Masters\datasets\resolution_summary.csv


,series,zone,n_60min,n_15min,first_60min,last_60min,first_15min,last_15min,overlap_stamps,pct_hours_varying,pct_active_varying,pct_hours_all_zero,first_varying_hour,first_sustained_day,lag_days,n_distinct,pct_zero,mean,max
0,DK1 | generation_forecast [solar],DK1,80904,16800,2016-01-01 00:00:00+00:00,2025-04-07 23:00:00+00:00,2025-04-08 00:00:00+00:00,2025-10-01 23:45:00+00:00,0,85.0,92.4,8.0,2025-04-10 01:00:00+00:00,2025-04-10,2.0,28536,35.3,198.44,2368.3
1,DK1 | generation_forecast [wind_offshore],DK1,81096,16608,2016-01-01 00:00:00+00:00,2025-04-07 23:00:00+00:00,2025-04-08 00:00:00+00:00,2025-10-01 23:45:00+00:00,0,99.3,99.4,0.0,2025-04-09 22:00:00+00:00,2025-04-09,1.0,33695,0.1,491.10,1157.9
2,DK1 | generation_forecast [wind_onshore],DK1,80976,16629,2016-01-01 00:00:00+00:00,2025-04-07 23:00:00+00:00,2025-04-08 00:00:00+00:00,2025-10-01 23:45:00+00:00,0,99.4,99.4,0.0,2025-04-09 00:00:00+00:00,2025-04-09,1.0,48439,0.0,879.06,3490.0
3,DK1 | load_forecast,DK1,85488,0,2016-01-01 00:00:00+00:00,2025-10-01 23:00:00+00:00,NaT,NaT,0,NaN,NaN,NaN,NaT,NaT,NaN,15340,0.0,2403.11,3943.0
4,DK1 | price,DK1,85463,192,2015-12-31 23:00:00+00:00,2025-09-30 21:00:00+00:00,2025-09-30 22:00:00+00:00,2025-10-02 21:45:00+00:00,0,100.0,100.0,0.0,2025-09-30 22:00:00+00:00,2025-09-30,0.0,20860,0.2,70.71,936.3
5,DK2 | generation_forecast [solar],DK2,80952,16792,2016-01-01 00:00:00+00:00,2025-04-07 23:00:00+00:00,2025-04-08 00:00:00+00:00,2025-10-01 23:45:00+00:00,0,80.5,92.2,12.7,2025-04-10 03:00:00+00:00,2025-04-10,2.0,23852,38.3,88.36,876.7
6,DK2 | generation_forecast [wind_offshore],DK2,81120,16608,2016-01-01 00:00:00+00:00,2025-04-07 23:00:00+00:00,2025-04-08 00:00:00+00:00,2025-10-01 23:45:00+00:00,0,99.1,99.1,0.1,2025-04-09 22:00:00+00:00,2025-04-09,1.0,31619,0.1,239.05,920.8
7,DK2 | generation_forecast [wind_onshore],DK2,81000,16608,2016-01-01 00:00:00+00:00,2025-04-07 23:00:00+00:00,2025-04-08 00:00:00+00:00,2025-10-01 23:45:00+00:00,0,98.0,98.0,0.0,2025-04-09 22:00:00+00:00,2025-04-09,1.0,18094,0.0,178.30,810.2
8,DK2 | load_forecast,DK2,85488,0,2016-01-01 00:00:00+00:00,2025-10-01 23:00:00+00:00,NaT,NaT,0,NaN,NaN,NaN,NaT,NaT,NaN,11739,0.0,1529.06,2559.0
9,DK2 | price,DK2,85463,192,2015-12-31 23:00:00+00:00,2025-09-30 21:00:00+00:00,2025-09-30 22:00:00+00:00,2025-10-02 21:45:00+00:00,0,100.0,100.0,0.0,2025-09-30 22:00:00+00:00,2025-09-30,0.0,20608,0.1,70.52,936.3


## 6. Cleaning — build the hourly panel

Steps, in order:

1. **Drop duplicate rows** (`drop_duplicates` on the full key).
2. **Drop the FI, EE, LT and LV zones.**
3. **Drop constant and near-constant series** — anything more than 90% exactly zero, the same
   threshold applied in section 2.
4. **Find the cut-off.** For each surviving zone, the last hour whose day-ahead price is still genuinely
   hourly: either a `PT60M` stamp, or a `PT15M` hour whose four values are identical. The cut-off is the
   **earliest** of those across zones, so no zone contributes a partial-15-minute tail.
5. **Truncate** every series at that cut-off (inclusive).
6. **Collapse to hourly** — mean of the quarter-hours within each clock hour. For prices in the retained
   window the four values are identical, so the mean is exact and not an approximation; for MW quantities the
   mean is the correct hourly average.
7. **Save** in the same long schema as the raw file.

In [32]:
DROP_ZONES = ["FI", "EE", "LT", "LV"]


def last_hourly_price_stamp(d: pd.DataFrame) -> pd.Series:
    """Per zone: the last clock hour whose day-ahead price is still purely hourly.

    A PT60M stamp always qualifies. A PT15M hour qualifies only while its four values are
    identical, i.e. before the zone's first genuinely varying hour.
    """
    p = d[d.variable == "price"].copy()
    p["hour"] = p.timestamp_utc.dt.floor("h")
    out = {}
    for z, g in p.groupby("zone"):
        last60 = g.loc[g.resolution == "PT60M", "hour"].max()
        g15 = g[g.resolution == "PT15M"]
        if g15.empty:
            out[z] = last60
            continue
        a = g15.groupby("hour")["value"].agg(nun="nunique", cnt="count")
        a = a[a.cnt == 4]
        varying = a.index[a.nun > 1]
        last15 = (varying.min() - pd.Timedelta(hours=1)) if len(varying) else a.index.max()
        out[z] = max(x for x in [last60, last15] if pd.notna(x))
    return pd.Series(out).sort_values()


def clean(raw_df: pd.DataFrame, drop_zones=DROP_ZONES, verbose=True):
    log = lambda *a: print(*a) if verbose else None
    n0 = len(raw_df)

    # 1. duplicates
    d = raw_df.drop_duplicates(KEY, keep="first").copy()
    log(f"1. drop_duplicates      : {n0:,} -> {len(d):,}  (-{n0 - len(d):,})")

    # 2. zones
    n = len(d)
    d = d[~d.zone.isin(drop_zones)].copy()
    log(f"2. drop {'/'.join(drop_zones):<15}: {n:,} -> {len(d):,}  (-{n - len(d):,})")

    # 3. constant and near-constant series (same threshold as section 2)
    n = len(d)
    d, dropped = drop_near_constant(d, verbose=False)
    log(f"3. drop >{MAX_PCT_ZERO:g}% zero series: {n:,} -> {len(d):,}  (-{n - len(d):,})")
    for s in dropped:
        log(f"     dropped {s}")

    # 4. cut-off
    last_hourly = last_hourly_price_stamp(d)
    cutoff = last_hourly.min()
    log("\n4. last purely-hourly price stamp per zone:")
    for z, t in last_hourly.items():
        log(f"     {z:<4} {t}")
    log(f"   cut-off (earliest)   : {cutoff}  [inclusive]")

    # 5. truncate
    n = len(d)
    d = d[d.timestamp_utc <= cutoff].copy()
    log(f"5. truncate at cut-off  : {n:,} -> {len(d):,}  (-{n - len(d):,})")

    # 6. collapse to hourly
    d["timestamp_utc"] = d.timestamp_utc.dt.floor("h")
    group = ["zone", "eic", "variable", "document_type", "process_type", "psr_type",
             "business_type", "curve_type", "contract_type", "unit", "currency", "timestamp_utc"]
    group = [c for c in group if c in d.columns]
    n = len(d)
    hourly = d.groupby(group, as_index=False, observed=True)["value"].mean()
    hourly["resolution"] = "PT60M"
    hourly = hourly[[c for c in raw_df.columns if c in hourly.columns]].sort_values(
        ["zone", "variable", "psr_type", "timestamp_utc"]).reset_index(drop=True)
    log(f"6. collapse to hourly   : {n:,} -> {len(hourly):,}  (-{n - len(hourly):,})")

    return hourly, {"cutoff": cutoff, "last_hourly_price": last_hourly, "dropped_series": dropped}


clean_df, info = clean(raw)

1. drop_duplicates      : 5,883,021 -> 5,837,301  (-45,720)
2. drop FI/EE/LT/LV    : 5,837,301 -> 4,314,611  (-1,522,690)
3. drop >90% zero series: 4,314,611 -> 3,493,383  (-821,228)
     dropped NO1 | generation_forecast [solar]
     dropped NO1 | generation_forecast [wind_offshore]
     dropped NO2 | generation_forecast [solar]
     dropped NO2 | generation_forecast [wind_offshore]
     dropped NO3 | generation_forecast [solar]
     dropped NO3 | generation_forecast [wind_offshore]
     dropped NO4 | generation_forecast [solar]
     dropped NO4 | generation_forecast [wind_offshore]
     dropped NO5 | generation_forecast [solar]
     dropped NO5 | generation_forecast [wind_offshore]
     dropped NO5 | generation_forecast [wind_onshore]

4. last purely-hourly price stamp per zone:
     DK1  2025-09-30 21:00:00+00:00
     DK2  2025-09-30 21:00:00+00:00
     NO1  2025-09-30 21:00:00+00:00
     NO2  2025-09-30 21:00:00+00:00
     NO3  2025-09-30 21:00:00+00:00
     NO4  2025-09-30 21:00:0

In [33]:
# Checks before saving.
c = clean_df.copy()
c["series"] = c.zone + " | " + c.variable + np.where(c.psr_type != "", " [" + c.psr_type + "]", "")

assert c.resolution.eq("PT60M").all(), "non-hourly rows survived"
assert not c.duplicated(["series", "timestamp_utc"]).any(), "duplicate (series, timestamp)"
assert c.timestamp_utc.max() == info["cutoff"], "cut-off not applied"
assert not c.zone.isin(DROP_ZONES).any(), "dropped zone survived"
assert c.groupby("series")["value"].nunique().min() > 1, "constant series survived"

print(f"{len(c):,} rows | {c.zone.nunique()} zones | {c.series.nunique()} series")
print(f"span: {c.timestamp_utc.min()} -> {c.timestamp_utc.max()}")
print("\nsurviving series and their coverage:")
cov = c.groupby("series").agg(rows=("value", "size"),
                              first=("timestamp_utc", "min"),
                              last=("timestamp_utc", "max"))
expected = ((cov["last"] - cov["first"]).dt.total_seconds() / 3600 + 1).astype(int)
cov["pct_complete"] = (100 * cov.rows / expected).round(1)
display(cov)

3,207,707 rows | 11 zones | 40 series
span: 2015-12-31 23:00:00+00:00 -> 2025-09-30 21:00:00+00:00

surviving series and their coverage:


,rows,first,last,pct_complete
series,,,,
DK1 | generation_forecast [solar],85078,2016-01-01 00:00:00+00:00,2025-09-30 21:00:00+00:00,99.6
DK1 | generation_forecast [wind_offshore],85222,2016-01-01 00:00:00+00:00,2025-09-30 21:00:00+00:00,99.7
DK1 | generation_forecast [wind_onshore],85108,2016-01-01 00:00:00+00:00,2025-09-30 21:00:00+00:00,99.6
DK1 | load_forecast,85462,2016-01-01 00:00:00+00:00,2025-09-30 21:00:00+00:00,100.0
DK1 | price,85463,2015-12-31 23:00:00+00:00,2025-09-30 21:00:00+00:00,100.0
DK2 | generation_forecast [solar],85124,2016-01-01 00:00:00+00:00,2025-09-30 21:00:00+00:00,99.6
DK2 | generation_forecast [wind_offshore],85246,2016-01-01 00:00:00+00:00,2025-09-30 21:00:00+00:00,99.7
DK2 | generation_forecast [wind_onshore],85126,2016-01-01 00:00:00+00:00,2025-09-30 21:00:00+00:00,99.6
DK2 | load_forecast,85462,2016-01-01 00:00:00+00:00,2025-09-30 21:00:00+00:00,100.0


In [34]:
# Sanity check: prices are identical to the raw hourly values, not smeared by the aggregation.
chk = (raw.drop_duplicates(KEY)
          .query("zone == 'NO1' and variable == 'price' and resolution == 'PT15M'")
          .assign(hour=lambda x: x.timestamp_utc.dt.floor("h"))
          .groupby("hour")["value"].agg(["min", "max"]))
chk = chk[chk.index <= info["cutoff"]]
print("NO1 15-min price hours retained after the cut-off where the 4 values were NOT identical:",
      int((chk["max"] - chk["min"] > 1e-9).sum()))
print("-> 0 means the hourly collapse of Norwegian prices is exact, not an average.")

NO1 15-min price hours retained after the cut-off where the 4 values were NOT identical: 0
-> 0 means the hourly collapse of Norwegian prices is exact, not an average.


In [17]:
clean_df.to_parquet(OUT, index=False)
print(f"written: {OUT.resolve()}  ({OUT.stat().st_size / 1e6:.1f} MB)")
clean_df.head()

written: /home/claude/nbtest/datasets/nordic_baltic_clean_hourly.parquet  (20.5 MB)


,zone,eic,variable,document_type,process_type,timestamp_utc,value,resolution,psr_type,business_type,curve_type,contract_type,unit,currency
0,DK1,10YDK-1--------W,generation_forecast,A69,A01,2016-01-01 00:00:00+00:00,0.0,PT60M,B16,A94,A03,,MAW,
1,DK1,10YDK-1--------W,generation_forecast,A69,A01,2016-01-01 01:00:00+00:00,0.0,PT60M,B16,A94,A03,,MAW,
2,DK1,10YDK-1--------W,generation_forecast,A69,A01,2016-01-01 02:00:00+00:00,0.0,PT60M,B16,A94,A03,,MAW,
3,DK1,10YDK-1--------W,generation_forecast,A69,A01,2016-01-01 03:00:00+00:00,0.0,PT60M,B16,A94,A03,,MAW,
4,DK1,10YDK-1--------W,generation_forecast,A69,A01,2016-01-01 04:00:00+00:00,0.0,PT60M,B16,A94,A03,,MAW,


## 7. Notes for the modelling stage

- The retained window ends at the cut-off printed above, so the price panel is **uniformly hourly** —
  24 observations per zone per day, no mixed-resolution days.
- The two days of genuine 15-minute prices (1–2 Oct 2025) are deliberately outside the window. They are far
  too short to model and would otherwise give Norway 96 rows a day against everyone else's 24.
- `NO5 | generation_forecast [wind_onshore]` and `NO2 | generation_forecast [wind_offshore]` are not exactly constant but
  are 95–99% zero, so section 2 drops them along with the exactly-constant ones.
  `NO2 | generation_forecast [wind_offshore]` was also only 52% complete over its span (a 20-month gap from
  Jan 2024 to Sep 2025); every series that survives is 99.6–100% complete.
- Load and generation forecasts for DK1, DK2 and SE1–SE4 are natively hourly for the whole retained window,
  so nothing is aggregated for them — only the Norwegian series from March/April 2025 onward are collapsed.
- `psr_type` in the saved parquet holds `solar` / `wind_onshore` / `wind_offshore`, not the raw
  ENTSO-E `B16` / `B19` / `B18` codes. The mapping is `PSR_NAMES` in section 0.
